# InstructABSA — Train + Evaluate trên Kaggle T4

Notebook này huấn luyện InstructABSA (ATSC subtask, instruction Set-2) và đánh giá trên 3 dataset: SemEval-14 Restaurant, SemEval-14 Laptop, UIT-VSFC. Dùng cùng phần cứng (1x T4 16GB) như notebook 01 LCF-BERT để so sánh latency công bằng.

**Quy trình:**
1. Upload `instruct_absa_bundle.zip` lên Kaggle dưới dạng Dataset (hoặc đính kèm trực tiếp).
2. Giải nén vào `/kaggle/working/absa-sota-survey/`.
3. Cài deps tối thiểu.
4. Train 3 model (rest → lap → vsfc), checkpoint lưu vào `/kaggle/working/absa-sota-survey/checkpoints/`.
5. Evaluate 3 model qua `evaluate.py --given-aspect`.
6. Đóng gói `results/` thành zip — tự xuất hiện ở tab **Output** của Kaggle để tải về.

**Backbone:**
- EN: `allenai/tk-instruct-base-def-pos` (220M)
- VI: `google/mt5-base` (580M)

**Hardware:** Kaggle T4 16GB, fp16, batch 4 (EN) / 2 (VI), grad accum 4/8.

**Ghi chú:** Kaggle session lasts ~9-12h, `/kaggle/working/` persistent trong session. Không cần mount Drive.

## 1. Kiểm tra GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')

## 2. Upload bundle

**Cách 1 — Upload qua Kaggle Dataset (khuyến nghị, persistent):**
1. Tab phải → **Add Input** → **Upload** → chọn `instruct_absa_bundle.zip` từ máy local.
2. Đặt tên dataset (vd: `instruct-absa-bundle`).
3. Sau khi upload, file sẽ xuất hiện ở `/kaggle/input/instruct-absa-bundle/instruct_absa_bundle.zip`.

**Cách 2 — Upload tạm qua working dir** (mỗi session phải upload lại): kéo thả vào tab Files, file vào `/kaggle/working/instruct_absa_bundle.zip`.

Cell dưới tự dò 2 vị trí:

In [ ]:
import os, glob, zipfile, shutil

# Kaggle dataset có thể ở 2 dạng:
#   (A) File zip:  /kaggle/input/.../instruct_absa_bundle.zip
#   (B) Đã giải nén sẵn (Kaggle tự bung zip khi tạo Dataset):
#       /kaggle/input/.../<dataset>/{evaluate.py, models/, configs/, ...}
WORK_DIR = '/kaggle/working/absa-sota-survey'
os.makedirs(WORK_DIR, exist_ok=True)

# (A) tìm zip
zip_candidates = (
    glob.glob('/kaggle/input/**/instruct_absa_bundle.zip', recursive=True)
    + glob.glob('/kaggle/input/**/*.zip', recursive=True)
    + ['/kaggle/working/instruct_absa_bundle.zip']
)
zip_path = next((p for p in zip_candidates if os.path.exists(p)), None)

if zip_path:
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(WORK_DIR)
    print('Extracted zip ->', WORK_DIR)
else:
    # (B) tìm thư mục dataset chứa evaluate.py (sentinel file)
    sentinels = glob.glob('/kaggle/input/**/evaluate.py', recursive=True)
    assert sentinels, 'Không tìm thấy evaluate.py trong /kaggle/input/. Kiểm tra Add Input.'
    src_root = os.path.dirname(sentinels[0])
    print('Found extracted dataset at:', src_root)
    # Copy toàn bộ cây sang /kaggle/working (writable) để train script ghi checkpoint được
    for entry in os.listdir(src_root):
        src = os.path.join(src_root, entry)
        dst = os.path.join(WORK_DIR, entry)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
    print('Copied dataset ->', WORK_DIR)

%cd /kaggle/working/absa-sota-survey
!ls

## 3. Cài đặt thư viện tối thiểu

Kaggle đã có torch + numpy. Chỉ cần update transformers + sentencepiece (cho mT5).

In [ ]:
!pip install -q "transformers>=4.45,<5.0" accelerate sentencepiece pyyaml protobuf

## 4. Train — SemEval-14 Restaurant (EN)

Tk-Instruct-base-def-pos 220M, batch 4, grad accum 4, 4 epochs, fp16. Ước lượng ~15-20 phút.

In [ ]:
!python -m models.instruct_absa.train --config configs/instruct_absa_en_restaurant.yaml

## 5. Train — SemEval-14 Laptop (EN)

In [ ]:
!python -m models.instruct_absa.train --config configs/instruct_absa_en_laptop.yaml

## 6. Train — UIT-VSFC (VI)

mT5-base 580M, batch 2, grad accum 8, 4 epochs. Ước lượng ~30-40 phút (data lớn nhất + model nặng nhất).

In [ ]:
!python -m models.instruct_absa.train --config configs/instruct_absa_vi.yaml

## 7. Evaluate — Restaurant (EN)

In [ ]:
!python evaluate.py \
    --predictor predictors.instruct_absa:InstructABSAPredictor \
    --predictor-kwargs '{"checkpoint":"checkpoints/semeval14_rest/instruct_absa_best","language":"en"}' \
    --test-set data/processed/lcf_bert/semeval14_rest_test.jsonl \
    --given-aspect --output-dir results

## 8. Evaluate — Laptop (EN)

In [ ]:
!python evaluate.py \
    --predictor predictors.instruct_absa:InstructABSAPredictor \
    --predictor-kwargs '{"checkpoint":"checkpoints/semeval14_lap/instruct_absa_best","language":"en"}' \
    --test-set data/processed/lcf_bert/semeval14_lap_test.jsonl \
    --given-aspect --output-dir results

## 9. Evaluate — UIT-VSFC (VI)

In [ ]:
!python evaluate.py \
    --predictor predictors.instruct_absa:InstructABSAPredictor \
    --predictor-kwargs '{"checkpoint":"checkpoints/vsfc/instruct_absa_best","language":"vi"}' \
    --test-set data/processed/lcf_bert/vsfc_test.jsonl \
    --given-aspect --output-dir results

## 10. Đóng gói results để tải về

File zip xuất hiện ở `/kaggle/working/instruct_absa_results.zip` → tab **Output** bên phải có nút download.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/instruct_absa_results', 'zip', '/kaggle/working/absa-sota-survey/results')
print('Created /kaggle/working/instruct_absa_results.zip')
!ls -lh /kaggle/working/instruct_absa_results.zip

## 11. (Tuỳ chọn) In nhanh metrics ngay tại Kaggle

In [ ]:
import json, glob
for p in sorted(glob.glob('/kaggle/working/absa-sota-survey/results/metrics/instruct_absa_*.json')):
    m = json.load(open(p))
    print(f"{m['method']:14s} {m['dataset']:30s} "
          f" sent_acc={m['sentiment_accuracy']:.4f}"
          f"  macro_f1={m['sentiment_macro_f1']:.4f}"
          f"  latency={m['avg_latency_ms']}ms")